<a href="https://colab.research.google.com/github/asryan11/food_rep/blob/main/food_rep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tensorflow
!pip install nltk
!pip install pandas
!pip install numpy
!pip install sklearn
#!pip install tensorflow_decision_forests

  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [ ]:
import pandas as pd
import numpy as np
import nltk
import csv
from nltk import word_tokenize
from nltk.corpus import stopwords
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import Dense, LSTM, Embedding
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tensorflow import keras
from tensorflow.keras import layers
#import tensorflow_decision_forests as tfdf
import tensorflow as tf


In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

In [ ]:
df= pd.read_csv('/content/drive/MyDrive/ML Projects/Food Recipe LLM/Cleaned_Indian_Food_Dataset.csv')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5938 entries, 0 to 5937
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   TranslatedRecipeName    5938 non-null   object
 1   TranslatedIngredients   5938 non-null   object
 2   TotalTimeInMins         5938 non-null   int64 
 3   Cuisine                 5938 non-null   object
 4   TranslatedInstructions  5938 non-null   object
 5   URL                     5938 non-null   object
 6   Cleaned-Ingredients     5938 non-null   object
 7   image-url               5938 non-null   object
 8   Ingredient-count        5938 non-null   int64 
dtypes: int64(2), object(7)
memory usage: 417.6+ KB


In [ ]:
df.isnull().sum()

,0
TranslatedRecipeName,0
TranslatedIngredients,0
TotalTimeInMins,0
Cuisine,0
TranslatedInstructions,0
URL,0
Cleaned-Ingredients,0
image-url,0
Ingredient-count,0


In [ ]:
train = df.drop(['TranslatedIngredients', 'TotalTimeInMins', 'Cuisine', 'URL', 'image-url'], axis=1)

In [ ]:
# lem= WordNetLemmatizer()
# for i,row in df.iterrows():
#   filter_sentence=[]
#   sentence=row['directions']
#   sentence=re.sub(r'[^\w\s]','',sentence)
#   words=nltk.word_tokenize(sentence)
#   words=[w for w in words if not w in stop_words]
#   for w in words:
#     filter_sentence.append(lem.lemmatize(w))
#   print(filter_sentence)
#   df.loc[i,'directions']=' '.join(filter_sentence)

In [ ]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5938 entries, 0 to 5937
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   TranslatedRecipeName    5938 non-null   object
 1   TranslatedInstructions  5938 non-null   object
 2   Cleaned-Ingredients     5938 non-null   object
 3   Ingredient-count        5938 non-null   int64 
dtypes: int64(1), object(3)
memory usage: 185.7+ KB


In [ ]:
# train = train.sample(frac = 1)

In [ ]:
train.head()

,TranslatedRecipeName,TranslatedInstructions,Cleaned-Ingredients,Ingredient-count
0,Masala Karela Recipe,"To begin making the Masala Karela Recipe,de-se...","salt,amchur (dry mango powder),karela (bitter ...",10
1,Spicy Tomato Rice (Recipe),"To make tomato puliogere, first cut the tomato...","tomato,salt,chickpea lentils,green chilli,rice...",12
2,Ragi Semiya Upma Recipe - Ragi Millet Vermicel...,"To begin making the Ragi Vermicelli Recipe, fi...","salt,rice vermicelli noodles (thin),asafoetida...",12
3,Gongura Chicken Curry Recipe - Andhra Style Go...,To begin making Gongura Chicken Curry Recipe f...,"tomato,salt,ginger,sorrel leaves (gongura),fen...",15
4,Andhra Style Alam Pachadi Recipe - Adrak Chutn...,"To make Andhra Style Alam Pachadi, first heat ...","tomato,salt,ginger,red chillies,curry,asafoeti...",12


In [ ]:
import re
from nltk.stem import WordNetLemmatizer
stop_words= set(stopwords.words('english'))

In [ ]:
# lem= WordNetLemmatizer()
# for i,row in train.iterrows():
#   filter_sentence=[]
#   sentence=row['TranslatedInstructions']
#   sentence=re.sub(r'[^\w\s]','',sentence)
#   words=nltk.word_tokenize(sentence)
#   words=[w for w in words if not w in stop_words]
#   for w in words:
#     filter_sentence.append(lem.lemmatize(w))
#   print(filter_sentence)
#   train.loc[i,'directions']=' '.join(filter_sentence)

In [ ]:
# lem= WordNetLemmatizer()
# for i,row in train.iterrows():
#   filter_sentence=[]
#   sentence=row['Cleaned-Ingredients']
#   sentence=re.sub(r'[^\w\s]','',sentence)
#   words=nltk.word_tokenize(sentence)
#   words=[w for w in words if not w in stop_words]
#   for w in words:
#     filter_sentence.append(lem.lemmatize(w))
#   print(filter_sentence)
#   train.loc[i,'ingredients']=' '.join(filter_sentence)

In [ ]:
# lem= WordNetLemmatizer()
# for i,row in train.iterrows():
#   filter_sentence=[]
#   sentence=row['TranslatedRecipeName']
#   sentence=re.sub(r'[^\w\s]','',sentence)
#   words=nltk.word_tokenize(sentence)
#   words=[w for w in words if not w in stop_words]
#   for w in words:
#     filter_sentence.append(lem.lemmatize(w))
#   print(filter_sentence)
#   train.loc[i,'recipe_name']=' '.join(filter_sentence)

In [ ]:
train.head()

,TranslatedRecipeName,TranslatedInstructions,Cleaned-Ingredients,Ingredient-count
0,Masala Karela Recipe,"To begin making the Masala Karela Recipe,de-se...","salt,amchur (dry mango powder),karela (bitter ...",10
1,Spicy Tomato Rice (Recipe),"To make tomato puliogere, first cut the tomato...","tomato,salt,chickpea lentils,green chilli,rice...",12
2,Ragi Semiya Upma Recipe - Ragi Millet Vermicel...,"To begin making the Ragi Vermicelli Recipe, fi...","salt,rice vermicelli noodles (thin),asafoetida...",12
3,Gongura Chicken Curry Recipe - Andhra Style Go...,To begin making Gongura Chicken Curry Recipe f...,"tomato,salt,ginger,sorrel leaves (gongura),fen...",15
4,Andhra Style Alam Pachadi Recipe - Adrak Chutn...,"To make Andhra Style Alam Pachadi, first heat ...","tomato,salt,ginger,red chillies,curry,asafoeti...",12


In [ ]:
#train["ing"]= [train[ing].append('chicken') if i == 'Chicken' else 0 for i in train['TranslatedRecipeName']]
# for i in train['Cleaned-Ingredients']:
#   i == 'chicken'
#   train[ing].append('chicken')

In [ ]:
def extract_keywords(recipe_text):
  # Tokenization
  tokens = nltk.word_tokenize(recipe_text)

  # Remove stop words
  stop_words = set(stopwords.words('english'))
  filtered_tokens = [word for word in tokens if word not in stop_words]


  # Part-of-speech tagging
  tagged = nltk.pos_tag(filtered_tokens)

  # Extract nouns and adjectives
  keywords = [word for word, tag in tagged if tag.startswith('NN') or tag.startswith('JJ')]

  return keywords

all_keywords = []

# Call the function outside of the function definition to avoid recursion
for index, row in train.iterrows():
    text = row['TranslatedInstructions']
    keywords = extract_keywords(text)
    #print(f"Keywords for recipe {index}: {keywords}")
    train.at[index, 'TranslatedInstructions'] = ' '.join(keywords)
    all_keywords.append(keywords) # Append keywords for current row

# Assign the list of keyword lists to the 'Keywords' column
train['Keywords'] = all_keywords

In [ ]:
train.head()

,TranslatedRecipeName,TranslatedInstructions,Cleaned-Ingredients,Ingredient-count,Keywords
0,Masala Karela Recipe,Masala Karela Recipe de-seed karela slice skin...,"salt,amchur (dry mango powder),karela (bitter ...",10,"[Masala, Karela, Recipe, de-seed, karela, slic..."
1,Spicy Tomato Rice (Recipe),tomato first cut tomatoes mixer puree heat oil...,"tomato,salt,chickpea lentils,green chilli,rice...",12,"[tomato, first, cut, tomatoes, mixer, puree, h..."
2,Ragi Semiya Upma Recipe - Ragi Millet Vermicel...,Ragi Vermicelli Recipe first steam ragi vermic...,"salt,rice vermicelli noodles (thin),asafoetida...",12,"[Ragi, Vermicelli, Recipe, first, steam, ragi,..."
3,Gongura Chicken Curry Recipe - Andhra Style Go...,Gongura Chicken Curry Recipe ingredients small...,"tomato,salt,ginger,sorrel leaves (gongura),fen...",15,"[Gongura, Chicken, Curry, Recipe, ingredients,..."
4,Andhra Style Alam Pachadi Recipe - Adrak Chutn...,Andhra Style Alam Pachadi heat oil pan Add coo...,"tomato,salt,ginger,red chillies,curry,asafoeti...",12,"[Andhra, Style, Alam, Pachadi, heat, oil, pan,..."


In [ ]:
# # ohe= OneHotEncoder()
# from sklearn.preprocessing import LabelEncoder
# label_encoder = LabelEncoder()

In [ ]:
# # ohe.fit_transform(train[['recipe_name', 'ingredients', 'directions']]).toarray()
# train['TranslatedInstructions'] = label_encoder.fit_transform(train['TranslatedInstructions'])

In [ ]:
# # feature =ohe.fit_transform(train[['recipe_name', 'ingredients', 'directions']]).toarray()
# train['Cleaned-Ingredients'] = label_encoder.fit_transform(train['Cleaned-Ingredients'])

In [ ]:
# ohe.categories_
train.head()

,TranslatedRecipeName,TranslatedInstructions,Cleaned-Ingredients,Ingredient-count,Keywords
0,Masala Karela Recipe,Masala Karela Recipe de-seed karela slice skin...,"salt,amchur (dry mango powder),karela (bitter ...",10,"[Masala, Karela, Recipe, de-seed, karela, slic..."
1,Spicy Tomato Rice (Recipe),tomato first cut tomatoes mixer puree heat oil...,"tomato,salt,chickpea lentils,green chilli,rice...",12,"[tomato, first, cut, tomatoes, mixer, puree, h..."
2,Ragi Semiya Upma Recipe - Ragi Millet Vermicel...,Ragi Vermicelli Recipe first steam ragi vermic...,"salt,rice vermicelli noodles (thin),asafoetida...",12,"[Ragi, Vermicelli, Recipe, first, steam, ragi,..."
3,Gongura Chicken Curry Recipe - Andhra Style Go...,Gongura Chicken Curry Recipe ingredients small...,"tomato,salt,ginger,sorrel leaves (gongura),fen...",15,"[Gongura, Chicken, Curry, Recipe, ingredients,..."
4,Andhra Style Alam Pachadi Recipe - Adrak Chutn...,Andhra Style Alam Pachadi heat oil pan Add coo...,"tomato,salt,ginger,red chillies,curry,asafoeti...",12,"[Andhra, Style, Alam, Pachadi, heat, oil, pan,..."


In [ ]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5938 entries, 0 to 5937
Data columns (total 5 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   TranslatedRecipeName    5938 non-null   object
 1   TranslatedInstructions  5938 non-null   object
 2   Cleaned-Ingredients     5938 non-null   object
 3   Ingredient-count        5938 non-null   int64 
 4   Keywords                5938 non-null   object
dtypes: int64(1), object(4)
memory usage: 232.1+ KB


In [ ]:
# feature_labels = ohe.categories_

In [ ]:
# feature_labels = np.array(feature_labels).ravel()

In [ ]:
# pd.DataFrame(feature, columns=feature_labels)

In [ ]:
# prompt: Using dataframe train: train this dataframe using a neural network to build a chatbot.

# This task is too complex to be completed here. Building a chatbot using a neural network requires:

# 1. **Data Preprocessing**: Cleaning and preparing the text data (tokenization, stemming/lemmatization, etc.)
# 2. **Model Selection**: Choosing an appropriate neural network architecture (e.g., Seq2Seq, Transformer)
# 3. **Training**: Training the model on the preprocessed data
# 4. **Evaluation**: Assessing the model's performance on a held-out dataset
# 5. **Deployment**: Integrating the trained model into a chatbot application.

# This process involves extensive code and cannot be condensed into a single response.


In [ ]:
# def preprocess_text(text):
#     tokens = word_tokenize(text)
#     tokens = [word for word in tokens if word.isalpha()]
#     stop_words = set(stopwords.words('english'))
#     tokens = [word for word in tokens if not word in stop_words]
#     return ' '.join(tokens)

In [ ]:
# train = df.drop(['yield', 'rating', 'url', 'cuisine_path', 'img_src'], axis=1)
# train['in'] = train['ingredients'].apply(preprocess_text)

# print(df.head())

In [ ]:
# train['ti'] = train['timing'].apply(preprocess_text)
# train['nut'] = train['nutrition'].apply(preprocess_text)
# train['dir'] = train['directions'].apply(preprocess_text)
# train['rip'] = train['recipe_name'].apply(preprocess_text)

In [ ]:
# train.fillna(0)

In [ ]:
# from sklearn.preprocessing import LabelEncoder, OneHotEncoder
# label_encoder_x= LabelEncoder()
# train[:]= label_encoder_x.fit_transform(train[10])
# #Encoding for dummy variables
# onehot_encoder= OneHotEncoder(categorical_features= [0])
# x= onehot_encoder.fit_transform(x).toarray()

In [ ]:
# def create_vocabulary_from_dataframe(df, text_columns):
#   """Creates a vocabulary from multiple text columns in a pandas DataFrame.

#   Args:
#     df: A pandas DataFrame.
#     text_columns: A list of column names containing text data.

#   Returns:
#     A dictionary mapping words to indices.
#   """

#   words = []
#   for col in text_columns:
#     words.extend([word for sentence in df[col] for word in sentence.split()])
#   unique_words = set(words)
#   vocab = {word: i+1 for i, word in enumerate(unique_words)}  # Start indices from 1
#   vocab['<UNK>'] = 0
#   return vocab

# def encode_text_from_dataframe(df, text_columns, vocab):
#   """Encodes text data from multiple text columns in a pandas DataFrame into numerical sequences.

#   Args:
#     df: A pandas DataFrame.
#     text_columns: A list of column names containing text data.
#     vocab: A dictionary mapping words to indices.

#   Returns:
#     A list of lists of encoded sequences, where each inner list corresponds to a column.
#   """

#   encoded_data = []
#   for col in text_columns:
#     column_encoded = []
#     for sentence in df[col]:
#       encoded_sentence = [vocab.get(word, 0) for word in sentence.split()]  # Use 0 for <UNK>
#       column_encoded.append(encoded_sentence)
#     encoded_data.append(column_encoded)
#   return encoded_data
# # Example usage
# data = {'text1': ['rip'], 'text2': ['dir'], 'text3': ['nut'], 'text4': ['ti'], 'text5': ['in']}
# df = pd.DataFrame(data)

# text_columns = ['text5']
# t_c= ['text1']

# vocab = create_vocabulary_from_dataframe(df, text_columns)
# encoded_data = encode_text_from_dataframe(df, text_columns, vocab)

In [ ]:
# train.head()

In [ ]:
# model = Sequential([
#     Embedding(input_dim=10, output_dim=64, input_length=10000),
#     LSTM(128),
#     Dense(25, activation='softmax')
# ])

In [ ]:
from sklearn.model_selection import train_test_split

# Select features (X) and target variable (y)
x = train['Keywords']  # Use a list to select multiple columns
y = train['TranslatedInstructions']  # Assuming you want the same columns for the target

# Split the data
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42)

In [ ]:
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical

tokenizer = Tokenizer()
tokenizer.fit_on_texts(x_train)
x_train_sequences = tokenizer.texts_to_sequences(x_train)

In [ ]:
max_sequence_length = 1000
x_train_padded = tf.keras.preprocessing.sequence.pad_sequences(x_train_sequences, maxlen=max_sequence_length)

x_train_flattened = x_train_padded.reshape(x_train_padded.shape[0], -1)

In [ ]:
vocab_size = len(tokenizer.word_index) + 1
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
num_classes = len(label_encoder.classes_)

y_train_one_hot = to_categorical(y_train_encoded, num_classes=num_classes)

In [ ]:
model = keras.Sequential([
    keras.layers.Dense(3, activation='relu', input_shape=(x_train_flattened.shape[1],)), # Specify input shape to the first layer
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(num_classes, activation='softmax')
])

/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
print("Shape of x_train_flattened:", x_train_flattened)
print("Shape of y_train_one_hot:", y_train_one_hot.shape)

Shape of x_train_flattened: [[   0    0    0 ...  768  105  461]
 [   0    0    0 ...   72  105  461]
 [   0    0    0 ... 3470 1106  696]
 ...
 [   0    0    0 ...    1  297   65]
 [   0    0    0 ...    1  271  131]
 [   0    0    0 ...   21  220  131]]
Shape of y_train_one_hot: (4156, 4155)


In [ ]:
train.head()

,TranslatedRecipeName,TranslatedInstructions,Cleaned-Ingredients,Ingredient-count,Keywords
0,Masala Karela Recipe,Masala Karela Recipe de-seed karela slice skin...,"salt,amchur (dry mango powder),karela (bitter ...",10,"[Masala, Karela, Recipe, de-seed, karela, slic..."
1,Spicy Tomato Rice (Recipe),tomato first cut tomatoes mixer puree heat oil...,"tomato,salt,chickpea lentils,green chilli,rice...",12,"[tomato, first, cut, tomatoes, mixer, puree, h..."
2,Ragi Semiya Upma Recipe - Ragi Millet Vermicel...,Ragi Vermicelli Recipe first steam ragi vermic...,"salt,rice vermicelli noodles (thin),asafoetida...",12,"[Ragi, Vermicelli, Recipe, first, steam, ragi,..."
3,Gongura Chicken Curry Recipe - Andhra Style Go...,Gongura Chicken Curry Recipe ingredients small...,"tomato,salt,ginger,sorrel leaves (gongura),fen...",15,"[Gongura, Chicken, Curry, Recipe, ingredients,..."
4,Andhra Style Alam Pachadi Recipe - Adrak Chutn...,Andhra Style Alam Pachadi heat oil pan Add coo...,"tomato,salt,ginger,red chillies,curry,asafoeti...",12,"[Andhra, Style, Alam, Pachadi, heat, oil, pan,..."


In [ ]:
# x_sample = tfdf.keras.pd_dataframe_to_tf_dataset(train, label="TranslatedInstructions", max_num_classes=6000) # Set an explicit upper limit for the number of classes
# x_sam = tfdf.keras.pd_dataframe_to_tf_dataset(train, label="Cleaned-Ingredients", max_num_classes=6000) # Set an explicit upper limit for the number of classes

In [ ]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(x_train_flattened, y_train_one_hot, epochs=8000, batch_size=36)

Streaming output truncated to the last 5000 lines.
65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.0853 - loss: 5.6943
Epoch 651/5000
65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.0845 - loss: 5.6841
Epoch 652/5000
65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.0914 - loss: 5.6504
Epoch 653/5000
65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.0811 - loss: 5.7596
Epoch 654/5000
65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.0151 - loss: 6.9633
Epoch 655/5000
65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.0200 - loss: 6.4478
Epoch 656/5000
65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.0467 - loss: 5.9792
Epoch 657/5000
65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.0519 - loss: 5.9257
Epoch 658/5000
65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.0759 - loss: 5.7248
Epoch 659/5000
65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.0716 - loss: 5.7449
Epoch 660/5000
65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.0771 - loss: 5.7

KeyboardInterrupt: 

In [ ]:
# import tensorflow as tf
# from sklearn.preprocessing import LabelEncoder
# from tensorflow.keras.preprocessing.text import Tokenizer
# from tensorflow.keras.utils import to_categorical

# # ... (your existing code for label encoding) ...

# # Tokenize the text data
# tokenizer = Tokenizer()
# tokenizer.fit_on_texts(x_train)
# x_train_sequences = tokenizer.texts_to_sequences(x_train)
# x_test_sequences = tokenizer.texts_to_sequences(x_test)

# max_sequence_length = 1000  # Adjust based on your data analysis
# x_train_padded = tf.keras.preprocessing.sequence.pad_sequences(x_train_sequences, maxlen=max_sequence_length)
# x_test_padded = tf.keras.preprocessing.sequence.pad_sequences(x_test_sequences, maxlen=max_sequence_length)

# # Flatten sequences for Dense layers
# x_train_flattened = x_train_padded.reshape(x_train_padded.shape[0], -1)
# x_test_flattened = x_test_padded.reshape(x_test_padded.shape[0], -1)

# vocab_size = len(tokenizer.word_index) + 1
# label_encoder = LabelEncoder()
# y_train_encoded = label_encoder.fit_transform(y_train)
# num_classes = len(label_encoder.classes_)



# model_better = keras.Sequential([
#     keras.layers.Dense(16, input_shape=(x_train_flattened.shape[1],), activation='relu'), # Change input_shape to match x_train_flattened
#     keras.layers.Dense(32, activation='relu'),
#     keras.layers.Dense(32, activation='relu'),
#     keras.layers.Dense(num_classes, activation='softmax')  # Use num_classes for output layer
# ])

# # Compiling the model
# model_better.compile(optimizer='adam',
#                      loss=keras.losses.SparseCategoricalCrossentropy(),
#                      metrics=['accuracy'])

# # fitting the model
# model_better.fit(x_train_flattened, y_train_encoded, epochs=10, batch_size=8)

In [ ]:
# inp = "Chicken, Masala"
# # Tokenize the text data
# tokenizer = Tokenizer()
# tokenizer.fit_on_texts(inp)
# x_train_sequences = tokenizer.texts_to_sequences(inp)

# # Pad sequences to a uniform length
# max_sequence_length = 1000  # Or determine this based on your data
# x_train_padded = tf.keras.preprocessing.sequence.pad_sequences(x_train_sequences, maxlen=max_sequence_length)

In [ ]:
# predictions = model.predict(x_train_padded)
# predicted_indices = np.argmax(predictions, axis=1)  # Get the index of the predicted class

# # Assuming you have the label encoder from your training process:
# predicted_labels = label_encoder.inverse_transform(predicted_indices)

# print(predicted_labels)

In [ ]:
def pred(ing):
  tokenizer = Tokenizer()
  tokenizer.fit_on_texts(ing)
  x_train_sequences = tokenizer.texts_to_sequences(ing)
  max_sequence_length = 1000  # Or determine this based on your data
  x_train_padded = tf.keras.preprocessing.sequence.pad_sequences(x_train_sequences, maxlen=max_sequence_length)
  predictions = model.predict(x_train_padded)
  predicted_indices = np.argmax(predictions, axis=1)  # Get the index of the predicted class
  predicted_labels = label_encoder.inverse_transform(predicted_indices)
  return predicted_labels

In [ ]:
pred(['Masala Karela Recipe'])

In [ ]:
model.evaluate(x_train_flattened, y_train_one_hot)

In [ ]:
y_pred = model.predict(x_train_flattened)

In [ ]:
y_test

In [ ]:
# X_train = encode_text_from_dataframe(train, text_columns, vocab)
# # y_train = encode_text_from_dataframe(train, t_c, vocab)

In [ ]:
# model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
# model.fit(x_train, y_train_encoded, epochs=10, batch_size=64)

In [ ]:
# print(train['in'].dtype)
# print(train['rip'].dtype)